평가 기준은 [평가 기준 문서](docs/evaluation_criteria.md)를 참고하세요.

## 평가 개요

- 선택한 실험의 저장된 가중치를 불러와 심비대·폐부종·흉수 소견 탐지 성능을 평가합니다.
- 현재 평가 데이터는 `data/processed/pilot/pilot_test.csv`이며, 평가 이미지 수는 해당 CSV를 기준으로 결정됩니다.
- 학습과 동일한 전처리를 적용하고, 전체 테스트 이미지의 정답·예측 확률·예측값을 수집합니다.

## 비교 실험

| 실험 | 모델 | 불확실 라벨 처리 | 비교 목적 |
| --- | --- | --- | --- |
| ① 기준 | DenseNet121 | U-multiclass | 기준 실험 |
| ② 모델 비교 | ResNet50 | U-multiclass | ①과 비교하여 모델 구조의 효과 확인 |
| ③ 라벨 처리 비교 | DenseNet121 | U-ignore | ①과 비교하여 불확실 라벨 처리 방식의 효과 확인 |

모델 종류 비교는 ①과 ②, 라벨 처리 방식 비교는 ①과 ③을 대상으로 합니다.

## 공통 평가 기준

- 세 실험은 동일한 테스트 데이터와 전처리 설정으로 비교합니다.
- 소견별 이진 평가에서는 해당 소견의 정답이 음성(0)·양성(1)인 항목만 사용합니다.
- 단일·복합 소견별 평가에서는 세 소견의 정답이 모두 음성·양성인 이미지만 사용합니다.
- 공통 양성 점수는 U-multiclass의 경우 `양성 확률 / (음성 확률 + 양성 확률)`, U-ignore의 경우 sigmoid로 얻은 양성 확률을 사용합니다.
- 공통 이진 판정 기준값의 기본값은 `0.5`이며, 양성 점수가 기준값 이상이면 양성으로 판정합니다. 기준값을 조정할 경우 검증 데이터에서 결정한 값을 적용합니다.
- AUROC·AP·AUPRC(사다리꼴)는 공통 양성 점수로, Precision·Recall·F1·민감도·특이도는 공통 이진 예측으로 계산합니다.
- U-multiclass의 3클래스 분류 보고서·혼동행렬과 정규화 전 양성 확률 AUROC는 부가 분석으로 제공합니다.

## 결과 저장 및 해석

- 각 실험을 선택하여 개별 평가하고, 예측 결과·평가 지표·시각화·오분류 사례·평가 설정을 저장합니다.
- 저장 위치: `evaluation/results/<실험 이름>/<학습 실행 폴더>/`
- 현재 노트북의 pilot 평가 결과는 최종 학습 모델의 평가 결과와 구분합니다. 최종 평가는 독립된 테스트 데이터에서 수행합니다.


## 1. 라이브러리 및 라벨 규칙

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

# 소견 순서
FINDINGS = ["Cardiomegaly", "Edema", "Pleural Effusion"]
FINDING_NAMES = ["심비대", "폐부종", "흉수"]

# 클래스 순서
CLASS_NAMES = ["음성", "양성", "불확실"]
CLASS_LABELS = [0, 1, 2]

## 2. 모델 예측 결과 불러오기

In [ ]:
import numpy as np
from pathlib import Path

# 노트북이 프로젝트의 evaluation 폴더에 있는 기준
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "evaluation":
    PROJECT_DIR = PROJECT_DIR.parent

IMAGE_ROOT = Path("/home/team30103/physionet_data")
CSV_DIR = PROJECT_DIR / "data" / "processed" / "pilot"

# 추가: 평가할 모델과 라벨 처리 방식
MODEL_NAME = "densenet121"
UNCERTAINTY_STRATEGY = "u_multiclass"

# 추가: 실험 이름
EXPERIMENT_NAME = f"{MODEL_NAME}_{UNCERTAINTY_STRATEGY}"

# 선택한 실험의 학습된 모델 경로
MODEL_PATH = (
    PROJECT_DIR / "runs" / "pilot"
    / "train_20261015_095921_874554"
    / "best_model.pth"
)

# 추가: 선택한 실험 확인
print("실험 이름:", EXPERIMENT_NAME)
print("모델 종류:", MODEL_NAME)
print("라벨 처리 방식:", UNCERTAINTY_STRATEGY)

print("프로젝트 경로:", PROJECT_DIR)
print("이미지 폴더 존재:", IMAGE_ROOT.is_dir())
print("test CSV 존재:", (CSV_DIR / "pilot_test.csv").is_file())
print("모델 파일 존재:", MODEL_PATH.is_file())

In [ ]:
import torch
from torch import nn
from torchvision.models import densenet121, resnet50

# GPU가 없으면 CPU 사용
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 선택한 모델·라벨 처리 방식에 맞는 구조
class CXRModel(nn.Module):
    def __init__(self, model_name, uncertainty_strategy):
        super().__init__()

        if uncertainty_strategy not in ("u_multiclass", "u_ignore"):
            raise ValueError(
                f"지원하지 않는 라벨 처리 방식: {uncertainty_strategy}"
            )

        self.uncertainty_strategy = uncertainty_strategy

        # U-multiclass: 3개 소견 × 3개 클래스
        # U-ignore: 3개 소견의 양성 여부
        output_size = (
            9 if uncertainty_strategy == "u_multiclass" else 3
        )

        if model_name == "densenet121":
            self.backbone = densenet121(weights=None)
            self.backbone.classifier = nn.Linear(
                self.backbone.classifier.in_features,
                output_size
            )

        elif model_name == "resnet50":
            self.backbone = resnet50(weights=None)
            self.backbone.fc = nn.Linear(
                self.backbone.fc.in_features,
                output_size
            )

        else:
            raise ValueError(
                f"지원하지 않는 모델: {model_name}"
            )

    def forward(self, images):
        logits = self.backbone(images)

        if self.uncertainty_strategy == "u_multiclass":
            return logits.reshape(-1, 3, 3)

        return logits


# 저장된 checkpoint 불러오기
checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

# 모든 실험에서 소견 순서는 같아야 함
assert checkpoint["target_labels"] == [
    "Cardiomegaly", "Edema", "Pleural Effusion"
], "체크포인트의 소견 순서가 평가 코드와 다릅니다."

# U-multiclass에서만 3클래스 순서 확인
if UNCERTAINTY_STRATEGY == "u_multiclass":
    assert checkpoint["class_names"] == [
        "negative", "positive", "uncertain"
    ], "체크포인트의 클래스 순서가 평가 코드와 다릅니다."

# 임시 데이터로 학습한 모델의 최종 평가 방지
if checkpoint.get("smoke_test", False):
    raise ValueError("임시 데이터로 학습한 demo 모델입니다.")

# 선택한 구조에 학습된 가중치 적용
model = CXRModel(
    model_name=MODEL_NAME,
    uncertainty_strategy=UNCERTAINTY_STRATEGY
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model.to(device)
model.eval()

print("실험 이름:", EXPERIMENT_NAME)
print("사용 장치:", device)
print("저장 당시 epoch:", checkpoint["epoch"])
print("학습된 모델 불러오기 완료")

In [ ]:
import sys
import pandas as pd
from torch.utils.data import DataLoader

# 프로젝트의 전처리 코드 사용
sys.path.insert(0, str(PROJECT_DIR))
from cxr_preprocessing import CXRDataset

# test CSV 읽기
test_df = pd.read_csv(
    CSV_DIR / "pilot_test.csv",
    dtype={"dicom_id": "string"}
)

# IMAGE_ROOT 바로 아래에 이미지가 모여 있는 기준
test_df["image_path"] = test_df["dicom_id"].map(
    lambda image_id: str(IMAGE_ROOT / f"{image_id}.jpg")
)

# 이미지 누락 확인
missing = test_df["image_path"].map(
    lambda path: not Path(path).is_file()
)

if missing.any():
    raise FileNotFoundError(
        f"test 이미지 {missing.sum()}장 누락. "
        f"첫 누락 경로: {test_df.loc[missing, 'image_path'].iloc[0]}"
    )

# 학습과 동일한 전처리·라벨 변환 적용
test_dataset = CXRDataset(
    test_df,
    image_size=checkpoint["input"]["height"]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

print("test 이미지 수:", len(test_dataset))
print("test 배치 수:", len(test_loader))
print("test DataLoader 준비 완료")

In [ ]:
# 공통 이진 판정 기준
THRESHOLD = 0.5

true_batches = []
prob_batches = []
score_batches = []

model.eval()

with torch.inference_mode():
    for batch_idx, (images, labels) in enumerate(
        test_loader, start=1
    ):
        images = images.to(device)

        logits = model(images).float()

        if UNCERTAINTY_STRATEGY == "u_multiclass":
            # 소견별 음성·양성·불확실 확률
            probs = logits.softmax(dim=-1)

            # 공통 평가용 양성 점수
            # 양성 확률 / (음성 확률 + 양성 확률)과 동일
            # 음성·양성 logit만 사용해 수치적으로 안정하게 계산
            scores = logits[..., :2].softmax(dim=-1)[..., 1]

        else:
            # U-ignore: 소견별 양성 확률
            probs = logits.sigmoid()
            scores = probs

        true_batches.append(labels.cpu())
        prob_batches.append(probs.cpu())
        score_batches.append(scores.cpu())

        print(
            f"예측 진행: {batch_idx}/{len(test_loader)} 배치"
        )

# 전체 배치 결과 합치기
y_true = torch.cat(true_batches).numpy()
y_prob = torch.cat(prob_batches).numpy()
y_score = torch.cat(score_batches).numpy()

# 공통 평가용 음성·양성 예측
y_pred_binary = (
    y_score >= THRESHOLD
).astype(np.int64)

# 모델 자체의 예측 결과
if UNCERTAINTY_STRATEGY == "u_multiclass":
    y_pred = y_prob.argmax(axis=-1)
else:
    y_pred = y_pred_binary.copy()

print("정답 형태:", y_true.shape)
print("모델 확률 형태:", y_prob.shape)
print("모델 예측 형태:", y_pred.shape)
print("공통 양성 점수 형태:", y_score.shape)
print("공통 이진 예측 형태:", y_pred_binary.shape)

## 3. 입력 데이터 검증

In [ ]:
n_images = len(y_true)

# 1. 모든 실험에서 공통으로 사용하는 형태
assert y_true.shape == (n_images, 3)
assert y_pred.shape == (n_images, 3)
assert y_score.shape == (n_images, 3)
assert y_pred_binary.shape == (n_images, 3)

# 2. 테스트 정답은 음성·양성·불확실을 그대로 보존
assert np.isin(y_true, [0, 1, 2]).all()

# 3. 모델 확률과 공통 양성 점수의 유효성 확인
assert np.isfinite(y_prob).all()
assert ((y_prob >= 0) & (y_prob <= 1)).all()

assert np.isfinite(y_score).all()
assert ((y_score >= 0) & (y_score <= 1)).all()

# 4. 공통 이진 예측 확인
assert 0 <= THRESHOLD <= 1
assert np.isin(y_pred_binary, [0, 1]).all()

assert np.array_equal(
    y_pred_binary,
    (y_score >= THRESHOLD).astype(np.int64)
)

# 5. 라벨 처리 방식별 출력 검증
if UNCERTAINTY_STRATEGY == "u_multiclass":
    assert y_prob.shape == (n_images, 3, 3)

    # 각 소견의 음성·양성·불확실 확률 합은 1
    assert np.allclose(
        y_prob.sum(axis=-1),
        1.0,
        atol=1e-5
    )

    assert np.array_equal(
        y_pred,
        y_prob.argmax(axis=-1)
    )

elif UNCERTAINTY_STRATEGY == "u_ignore":
    assert y_prob.shape == (n_images, 3)

    # 소견별 양성 확률을 공통 점수로 사용
    assert np.allclose(y_score, y_prob)
    assert np.array_equal(y_pred, y_pred_binary)

else:
    raise ValueError(
        f"지원하지 않는 라벨 처리 방식: {UNCERTAINTY_STRATEGY}"
    )

print("평가 데이터 검증 완료")
print("실험 이름:", EXPERIMENT_NAME)
print("평가 이미지 수:", n_images)
print("공통 이진 판정 기준:", THRESHOLD)

## 4. 소견별 성능 평가

In [ ]:
# 공통 이진 평가에서 사용할 클래스
BINARY_CLASS_NAMES = ["음성", "양성"]
BINARY_CLASS_LABELS = [0, 1]

# 1. 세 실험에 공통으로 적용하는 이진 평가
for i, finding in enumerate(FINDING_NAMES):
    # 해당 소견의 정답이 음성·양성인 이미지만 선택
    mask = np.isin(y_true[:, i], [0, 1])

    true_binary = y_true[mask, i]
    pred_binary = y_pred_binary[mask, i]

    print(f"\n===== {finding}: 공통 이진 평가 =====")
    print("평가 이미지 수:", int(mask.sum()))
    print("불확실 정답 제외 수:", int((~mask).sum()))

    if len(true_binary) == 0:
        print("평가 가능한 음성·양성 정답이 없습니다.")
        continue

    print(classification_report(
        true_binary,
        pred_binary,
        labels=BINARY_CLASS_LABELS,
        target_names=BINARY_CLASS_NAMES,
        zero_division=0
    ))

# 2. U-multiclass의 3클래스 부가 분석
if UNCERTAINTY_STRATEGY == "u_multiclass":
    for i, finding in enumerate(FINDING_NAMES):
        print(f"\n===== {finding}: 3클래스 부가 분석 =====")

        print(classification_report(
            y_true[:, i],
            y_pred[:, i],
            labels=CLASS_LABELS,
            target_names=CLASS_NAMES,
            zero_division=0
        ))

In [ ]:
# 1. 세 실험에 공통으로 적용하는 이진 혼동행렬
for i, finding in enumerate(FINDING_NAMES):
    mask = np.isin(y_true[:, i], [0, 1])

    print(f"\n===== {finding}: 공통 이진 혼동행렬 =====")

    if not mask.any():
        print("평가 가능한 음성·양성 정답이 없습니다.")
        continue

    cm = confusion_matrix(
        y_true[mask, i],
        y_pred_binary[mask, i],
        labels=BINARY_CLASS_LABELS
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"실제 {name}" for name in BINARY_CLASS_NAMES
        ],
        columns=[
            f"예측 {name}" for name in BINARY_CLASS_NAMES
        ]
    )

    display(cm_df)

# 2. U-multiclass의 3클래스 혼동행렬
if UNCERTAINTY_STRATEGY == "u_multiclass":
    for i, finding in enumerate(FINDING_NAMES):
        cm = confusion_matrix(
            y_true[:, i],
            y_pred[:, i],
            labels=CLASS_LABELS
        )

        cm_df = pd.DataFrame(
            cm,
            index=[
                f"실제 {name}" for name in CLASS_NAMES
            ],
            columns=[
                f"예측 {name}" for name in CLASS_NAMES
            ]
        )

        print(f"\n===== {finding}: 3클래스 부가 분석 =====")
        display(cm_df)

In [ ]:
# AUROC 평가
auroc_results = []

for i, finding in enumerate(FINDING_NAMES):
    # 해당 소견의 불확실 정답 제외
    mask = np.isin(y_true[:, i], [0, 1])

    true_binary = y_true[mask, i]
    positive_score = y_score[mask, i]

    common_auroc = np.nan
    raw_auroc = np.nan

    # 음성과 양성이 모두 있어야 계산 가능
    if len(np.unique(true_binary)) == 2:
        # 세 실험에 공통으로 적용하는 AUROC
        common_auroc = roc_auc_score(
            true_binary,
            positive_score
        )

        # U-multiclass의 원래 양성 확률 AUROC
        if UNCERTAINTY_STRATEGY == "u_multiclass":
            raw_auroc = roc_auc_score(
                true_binary,
                y_prob[mask, i, 1]
            )

    result = {
        "소견": finding,
        "평가 이미지 수": int(mask.sum()),
        "음성 수": int((true_binary == 0).sum()),
        "양성 수": int((true_binary == 1).sum()),
        "공통 AUROC": common_auroc,
    }

    # U-multiclass에서만 부가 지표 표시
    if UNCERTAINTY_STRATEGY == "u_multiclass":
        result["3클래스 양성 확률 AUROC (부가)"] = raw_auroc

    auroc_results.append(result)

auroc_df = pd.DataFrame(auroc_results)
display(auroc_df)

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

auprc_results = []

for i, finding in enumerate(FINDING_NAMES):
    # 해당 소견의 불확실 정답 제외
    mask = np.isin(y_true[:, i], [0, 1])

    true_binary = y_true[mask, i]
    positive_score = y_score[mask, i]

    auprc = np.nan

    # 음성과 양성이 모두 있을 때 계산
    if len(np.unique(true_binary)) == 2:
        precision, recall, _ = precision_recall_curve(
            true_binary,
            positive_score
        )

        # PR 곡선 아래 면적을 사다리꼴 방식으로 계산
        auprc = auc(recall, precision)

    auprc_results.append({
        "소견": finding,
        "평가 이미지 수": int(mask.sum()),
        "양성 비율": (
            float((true_binary == 1).mean())
            if len(true_binary) > 0 else np.nan
        ),
        "공통 AUPRC (사다리꼴)": auprc,
    })

auprc_df = pd.DataFrame(auprc_results)
display(auprc_df)

In [ ]:
from sklearn.metrics import average_precision_score

additional_results = []

for i, finding in enumerate(FINDING_NAMES):
    # 해당 소견의 불확실 정답 제외
    mask = np.isin(y_true[:, i], [0, 1])

    true = y_true[mask, i]
    pred = y_pred_binary[mask, i]
    positive_score = y_score[mask, i]

    # 공통 이진 예측으로 TP·FN·TN·FP 계산
    tp = np.sum((true == 1) & (pred == 1))
    fn = np.sum((true == 1) & (pred == 0))
    tn = np.sum((true == 0) & (pred == 0))
    fp = np.sum((true == 0) & (pred == 1))

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0 else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0 else np.nan
    )

    ap = np.nan

    # 기존과 동일하게 음성·양성이 모두 있을 때 AP 계산
    if len(np.unique(true)) == 2:
        ap = average_precision_score(
            true,
            positive_score
        )

    additional_results.append({
        "소견": finding,
        "평가 이미지 수": int(mask.sum()),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "AP": ap,
    })

additional_df = pd.DataFrame(additional_results)
display(additional_df)

## 5. 단일·복합 소견별 성능 비교

In [ ]:
# 세 소견의 정답이 모두 음성·양성인 이미지만 선택
certain_mask = np.isin(y_true, [0, 1]).all(axis=1)

# 이미지별 실제 양성 소견 개수
positive_count = (y_true == 1).sum(axis=1)

groups = {
    "양성 소견 0개": (
        certain_mask & (positive_count == 0)
    ),
    "단일 소견 (양성 1개)": (
        certain_mask & (positive_count == 1)
    ),
    "복합 소견 (양성 2~3개)": (
        certain_mask & (positive_count >= 2)
    ),
}

group_results = []

for group_name, mask in groups.items():
    n = int(mask.sum())

    if n > 0:
        # 공통 이진 예측과 실제 정답 비교
        matches = (
            y_true[mask] == y_pred_binary[mask]
        )

        # 이미지마다 세 소견을 모두 맞힌 비율
        exact_match = matches.all(axis=1).mean()

        # 전체 소견 판단 중 맞힌 비율
        finding_accuracy = matches.mean()

    else:
        exact_match = np.nan
        finding_accuracy = np.nan

    group_results.append({
        "그룹": group_name,
        "이미지 수": n,
        "세 소견 모두 정답 비율": exact_match,
        "소견별 판단 정확도": finding_accuracy,
    })

group_df = pd.DataFrame(group_results)
display(group_df)

## 6. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = PROJECT_DIR / "evaluation" / "MALGUN.TTF"
fm.fontManager.addfont(str(font_path))

plt.rcParams["font.family"] = fm.FontProperties(
    fname=str(font_path)
).get_name()
plt.rcParams["axes.unicode_minus"] = False

# 실험 이름과 학습 실행 폴더별로 결과 저장
SAVE_DIR = (
    PROJECT_DIR / "evaluation" / "results"
    / EXPERIMENT_NAME
    / MODEL_PATH.parent.name
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("실험 이름:", EXPERIMENT_NAME)
print("저장 폴더:", SAVE_DIR)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 공통 이진 혼동행렬
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, finding in enumerate(FINDING_NAMES):
    mask = np.isin(y_true[:, i], [0, 1])
    axes[i].set_title(finding)

    if not mask.any():
        axes[i].text(
            0.5, 0.5, "평가 가능한 정답 없음",
            ha="center", va="center",
            transform=axes[i].transAxes
        )
        axes[i].set_axis_off()
        continue

    cm = confusion_matrix(
        y_true[mask, i],
        y_pred_binary[mask, i],
        labels=BINARY_CLASS_LABELS
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        vmin=0,
        vmax=len(y_true),
        xticklabels=BINARY_CLASS_NAMES,
        yticklabels=BINARY_CLASS_NAMES,
        ax=axes[i]
    )

    axes[i].set_xlabel("예측")
    axes[i].set_ylabel("실제 정답")

fig.suptitle(f"{EXPERIMENT_NAME} | 공통 이진 혼동행렬")
fig.tight_layout(rect=[0, 0, 1, 0.92])

fig.savefig(
    SAVE_DIR / "confusion_matrix_binary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# 2. U-multiclass의 3클래스 혼동행렬
if UNCERTAINTY_STRATEGY == "u_multiclass":
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for i, finding in enumerate(FINDING_NAMES):
        cm = confusion_matrix(
            y_true[:, i],
            y_pred[:, i],
            labels=CLASS_LABELS
        )

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            vmin=0,
            vmax=len(y_true),
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            ax=axes[i]
        )

        axes[i].set_title(finding)
        axes[i].set_xlabel("예측")
        axes[i].set_ylabel("실제 정답")

    fig.suptitle(f"{EXPERIMENT_NAME} | 3클래스 부가 분석")
    fig.tight_layout(rect=[0, 0, 1, 0.92])

    fig.savefig(
        SAVE_DIR / "confusion_matrix_multiclass.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
ax = group_df.plot(
    x="그룹",
    y=["세 소견 모두 정답 비율", "소견별 판단 정확도"],
    kind="bar",
    figsize=(10, 5),
    rot=0,
    color=["steelblue", "orange"]
)

ax.set_title("양성 소견 개수에 따른 성능 비교")
ax.set_xlabel("그룹")
ax.set_ylabel("정확도")
ax.set_ylim(0, 1.1)

ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()

plt.savefig(
    SAVE_DIR / "finding_groups.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 7. 오분류 사례 분석

In [ ]:
# 이미지 ID와 예측 결과의 개수 확인
assert len(test_df) == len(y_true), (
    "테스트 CSV와 예측 결과의 이미지 수가 다릅니다."
)

error_results = []

for image_idx in range(len(y_true)):
    image_id = str(
        test_df.iloc[image_idx]["dicom_id"]
    )

    for finding_idx, finding in enumerate(FINDING_NAMES):
        true_label = int(
            y_true[image_idx, finding_idx]
        )

        # 해당 소견의 불확실 정답 제외
        if true_label not in BINARY_CLASS_LABELS:
            continue

        pred_label = int(
            y_pred_binary[image_idx, finding_idx]
        )

        # 틀린 경우만 기록
        if true_label != pred_label:
            error_results.append({
                "이미지 번호": image_idx + 1,
                "이미지 ID": image_id,
                "소견": finding,
                "실제 정답": BINARY_CLASS_NAMES[true_label],
                "모델 예측": BINARY_CLASS_NAMES[pred_label],
                "오류 유형": (
                    "FN" if true_label == 1 else "FP"
                ),
                "양성 점수": float(
                    y_score[image_idx, finding_idx]
                ),
            })

error_df = pd.DataFrame(
    error_results,
    columns=[
        "이미지 번호",
        "이미지 ID",
        "소견",
        "실제 정답",
        "모델 예측",
        "오류 유형",
        "양성 점수",
    ]
)

display(error_df)

In [ ]:
# 1. 소견별 오분류 개수
error_counts = (
    error_df.groupby("소견")
    .size()
    .reindex(FINDING_NAMES, fill_value=0)
    .rename("오분류 개수")
)

print("===== 소견별 오분류 개수 =====")
print(error_counts.to_string())

# 2. 소견별 오류 유형
error_types = (
    error_df.groupby(["소견", "실제 정답", "모델 예측"])
    .size()
    .reset_index(name="개수")
)

print("\n===== 소견별 오류 유형 =====")
print(error_types.to_string(index=False))

In [ ]:
import json
from datetime import datetime

# 앞서 지정한 시각화 저장 폴더와 동일한 경로
SAVE_DIR = (
    PROJECT_DIR / "evaluation" / "results"
    / EXPERIMENT_NAME
    / MODEL_PATH.parent.name
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)


# JSON 저장 함수
def save_json(filename, data):
    (SAVE_DIR / filename).write_text(
        json.dumps(
            data,
            ensure_ascii=False,
            indent=2,
            default=str
        ),
        encoding="utf-8"
    )


# 1. 정답·모델 출력·공통 평가용 결과·이미지 ID 저장
assert len(test_df) == len(y_true)

np.savez_compressed(
    SAVE_DIR / "predictions_test.npz",
    y_true=y_true,
    y_prob=y_prob,
    y_pred=y_pred,
    y_score=y_score,
    y_pred_binary=y_pred_binary,
    dicom_id=test_df["dicom_id"].astype(str).to_numpy(dtype=str)
)

# 2. 평가 지표 및 분석 결과 저장
tables = {
    "auroc": auroc_df,
    "auprc": auprc_df,
    "additional_metrics": additional_df,
    "finding_groups": group_df,
    "misclassifications": error_df,
}

for name, table in tables.items():
    table.to_csv(
        SAVE_DIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# 3. 공통 이진 분류 보고서 저장
binary_reports = {}

for i, finding in enumerate(FINDINGS):
    mask = np.isin(y_true[:, i], [0, 1])

    if mask.any():
        binary_reports[finding] = classification_report(
            y_true[mask, i],
            y_pred_binary[mask, i],
            labels=BINARY_CLASS_LABELS,
            target_names=BINARY_CLASS_NAMES,
            zero_division=0,
            output_dict=True
        )
    else:
        # 평가 가능한 정답이 없으면 JSON의 null로 기록
        binary_reports[finding] = None

save_json(
    "classification_reports_binary.json",
    binary_reports
)

# 4. U-multiclass의 3클래스 분류 보고서 저장
if UNCERTAINTY_STRATEGY == "u_multiclass":
    multiclass_reports = {
        finding: classification_report(
            y_true[:, i],
            y_pred[:, i],
            labels=CLASS_LABELS,
            target_names=CLASS_NAMES,
            zero_division=0,
            output_dict=True
        )
        for i, finding in enumerate(FINDINGS)
    }

    save_json(
        "classification_reports_multiclass.json",
        multiclass_reports
    )

# 5. 실험 및 평가 기준 기록
positive_score_rule = (
    "p_positive / (p_negative + p_positive)"
    if UNCERTAINTY_STRATEGY == "u_multiclass"
    else "sigmoid(logit)"
)

info = {
    "evaluated_at": datetime.now().astimezone().isoformat(),
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "uncertainty_strategy": UNCERTAINTY_STRATEGY,
    "model_path": str(MODEL_PATH),
    "test_csv": str(CSV_DIR / "pilot_test.csv"),
    "image_root": str(IMAGE_ROOT),
    "test_image_count": len(y_true),
    "saved_epoch": checkpoint["epoch"],
    "training_config": checkpoint.get("config"),
    "input_config": checkpoint.get("input"),
    "finding_order": FINDINGS,
    "ground_truth_label_codes": {
        "0": "음성",
        "1": "양성",
        "2": "불확실"
    },
    "native_class_order": (
        CLASS_NAMES
        if UNCERTAINTY_STRATEGY == "u_multiclass"
        else None
    ),
    "binary_class_order": BINARY_CLASS_NAMES,
    "positive_score_rule": positive_score_rule,
    "binary_threshold": float(THRESHOLD),
    "binary_prediction_rule": "y_score >= binary_threshold",
    "finding_evaluation_rule": (
        "해당 소견의 정답이 0 또는 1인 항목만 평가"
    ),
    "group_evaluation_rule": (
        "세 소견의 정답이 모두 0 또는 1인 이미지만 평가"
    ),
}

save_json("evaluation_info.json", info)

print("실험 이름:", EXPERIMENT_NAME)
print("평가 결과 저장 완료:", SAVE_DIR)